### Install Libraries

In [1]:
!pip install datasets
!pip install youtube-comment-downloader
!pip install youtube-search-python
!pip install "httpx<0.24.0"

  Attempting uninstall: httpcore
    Found existing installation: httpcore 1.0.2
    Uninstalling httpcore-1.0.2:
      Successfully uninstalled httpcore-1.0.2
  Attempting uninstall: httpx
    Found existing installation: httpx 0.27.0
    Uninstalling httpx-0.27.0:
      Successfully uninstalled httpx-0.27.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyterlab 4.2.5 requires httpx>=0.25.0, but you have httpx 0.23.3 which is incompatible.


In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets
import json
import pandas as pd

# === 1. Amazon Reviews (Meinungen, sehr gut für Sentiment) ===
# mlsum_all = load_dataset("mlsum", "de", split="train")
# mlsum_text = mlsum_all.map(lambda x: {"text": x["text"]}, remove_columns=mlsum_all.column_names)

# === 2. OpenSubtitles (Film-Dialoge, implizites Sentiment) ===
# subtitles = load_dataset("opus_books", "de-en", split="train[:20000]")
# subtitles = subtitles.map(lambda x: {"text": x["translation"]["de"]}, remove_columns=subtitles.column_names)


# === 3. SB10k (from Hugging Face) ===
sb10k = load_dataset("Alienmaster/SB10k", split="train")
print(sb10k.column_names)
sb10k = sb10k.map(lambda x: {
    "text": x["Text"],
    "label": x["Sentiment"].lower()  # Convert to 'positive', 'neutral', 'negative'
}, remove_columns=sb10k.column_names)

files.download(sb10k)

# === Kombinieren & Shuffle ===
combined_dataset = concatenate_datasets([sb10k])
combined_dataset = combined_dataset.shuffle(seed=42)

# === Speichern als JSONL ===
with open("german_sentiment_dataset.jsonl", "w", encoding="utf-8") as f:
    for example in combined_dataset:
        json.dump(example, f, ensure_ascii=False)
        f.write("\n")

print(f"✅ Fertig: {len(combined_dataset)} Texte gespeichert.")

['ID', 'Sentiment', 'Text', 'Normalized', 'POS-Tags', 'Dependency Labels', 'additional Annotations']


TypeError: stat: path should be string, bytes, os.PathLike or integer, not Dataset

### Download Youtube comment data

In [ ]:
import os
import time
from youtubesearchpython import VideosSearch
from youtube_comment_downloader import YoutubeCommentDownloader

# === Config ===
domains = {
    "gaming": ["gaming deutsch", "spiel bewertung", "pc spiele test", "konsole vergleich deutsch"],
    "music": ["deutschrap 2024", "musikvideo reaktion", "musik bewertung"],
    "products": ["produkttest deutsch", "smartphone review deutsch", "technik empfehlung"],
    "politics": ["nachrichten meinung deutsch", "politische debatte deutschland", "bundestag diskussion"],
    "news": ["tagesschau aktuell", "zdf heute meinung", "nachrichten kommentar"],
    "travel": ["hotel erfahrung deutsch", "reisebericht deutschland", "urlaub bewertung"]
}

videos_per_query = 50
max_comments_per_video = 1000
output_file = "youtube_comments_clean.txt"
processed_log = "processed_videos.log"
target_size_MB = 40  # Stop after 5MB

# === Load processed URLs ===
if os.path.exists(processed_log):
    with open(processed_log, "r", encoding="utf-8") as f:
        processed = set(line.strip() for line in f)
else:
    processed = set()

downloader = YoutubeCommentDownloader()

def file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024) if os.path.exists(path) else 0

with open(output_file, "a", encoding="utf-8") as out_file, open(processed_log, "a", encoding="utf-8") as log_file:
    for domain, queries in domains.items():
        for query in queries:
            print(f"\nSearching: {query}")
            try:
                videos_search = VideosSearch(query, limit=videos_per_query)
                results = videos_search.result()["result"]
            except Exception as e:
                print(f"Search failed: {e}")
                continue

            for video in results:
                url = video["link"]
                if url in processed:
                    print(f"Skipping: already processed {url}")
                    continue

                print(f"Processing: {video['title']}\n    URL: {url}")
                try:
                    count = 0
                    for comment in downloader.get_comments_from_url(url, sort_by=0):
                        text = comment.get("text", "").strip()
                        if text:
                            clean_text = text.replace("\r", " ").replace("\n", " ").strip()
                            out_file.write(clean_text + "\n")
                            count += 1
                            
                            if count >= max_comments_per_video:
                                break
                            if file_size_mb(output_file) >= target_size_MB:
                                print(f"\nReached {target_size_MB}MB — stopping.")
                                raise StopIteration

                    print(f"   Saved {count} comments.")
                    log_file.write(url + "\n")
                    log_file.flush()

                except StopIteration:
                    raise
                except Exception as e:
                    print(f"Comment scraping error: {e}")
                    continue

                time.sleep(1)


Searching: gaming deutsch
Skipping: already processed https://www.youtube.com/watch?v=5DnJbnMhXNM
Skipping: already processed https://www.youtube.com/watch?v=Kzlb526z-Jg
Processing: Endlich habe ich es fertig geterraformed... #MorizzGMC #minecraft
    URL: https://www.youtube.com/watch?v=YeXu47loaQQ
   ✔ Saved 16 comments.
Processing: Dieses Horrorspiel gibt mir einen Herzinfarkt...
    URL: https://www.youtube.com/watch?v=C5dsZZGVzsM
   ✔ Saved 626 comments.
Processing: Ich baue den besten Gaming PC der Welt
    URL: https://www.youtube.com/watch?v=yOvm9EoHRBE
   ✔ Saved 1000 comments.
Processing: Das LUSTIGSTE Game seit Lethal Company!
    URL: https://www.youtube.com/watch?v=fIo4o7YyFqI
   ✔ Saved 306 comments.
Processing: High-Tech BLITZER! 📸 - GTA 5 LSPD:FR #193 - Daniel Gaming - Deutsch
    URL: https://www.youtube.com/watch?v=V2q-mEuzLo8
   ✔ Saved 275 comments.
Processing: ASMR - ICH SPIELE CLASH ROYALE [GAMING] (GERMAN) | ASMR Tony
    URL: https://www.youtube.com/watch?v=Y2t